In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

# 从当前目录向上找 .git，定位根目录
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT/".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
# 把项目根目录加入到sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

from src.config import DB_PATH, FIGURES_DIR, ensure_dirs

ensure_dirs()

# 设置中文字体以及负号显示
from src.viz import apply_chinese, get_font_prop

font_prop = get_font_prop()
plt.rcParams["axes.unicode_minus"] = False


con = duckdb.connect(str(DB_PATH),read_only=True)


In [ ]:
user_funnel = con.sql("""

SELECT
    COUNT(DISTINCT CASE WHEN behavior_type = '浏览' THEN user_id END) AS pv_users,
    COUNT(DISTINCT CASE WHEN behavior_type = '收藏' THEN user_id END) AS fav_users,
    COUNT(DISTINCT CASE WHEN behavior_type = '加购' THEN user_id END) AS cart_users,
    COUNT(DISTINCT CASE WHEN behavior_type = '购买' THEN user_id END) AS buy_users
FROM user_behavior
""").df()
print(user_funnel)

In [ ]:
# 横向漏斗图
stages = ['浏览','收藏','加购','购买']
values = [
    user_funnel['pv_users'][0],
    user_funnel['fav_users'][0],
    user_funnel['cart_users'][0],
    user_funnel['buy_users'][0],
]

colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52"]
fig, ax = plt.subplots(figsize=(10,5))
bars = ax.barh(stages,values,color=colors)
ax.invert_yaxis()

for bar, v in zip(bars, values):
    ax.text(v,bar.get_y() + bar.get_height()/2,f"{v:,}",va="center",fontsize = 11)

ax.set_xlabel("用户数")
ax.set_title("用户行为漏斗")
apply_chinese(ax,font_prop)
plt.savefig(FIGURES_DIR/"user_funnel.png")
plt.show()

In [ ]:
# 转换率条形图
rates = {
    "收藏率/收藏转化": round(user_funnel["fav_users"][0] / user_funnel["pv_users"][0] * 100,2),
    "加购率/加购转化": round(user_funnel["cart_users"][0] / user_funnel["pv_users"][0] * 100,2),
    "整体转化率": round(user_funnel["buy_users"][0] / user_funnel["pv_users"][0] * 100,2),
}

fig,ax = plt.subplots(figsize=(8,4))
bars = ax.bar(rates.keys(),rates.values(),color = ["#4C72B0", "#55A868", "#DD8452", "#C44E52"])
for bar, v in zip(bars, rates.values()):
    ax.text(bar.get_x() + bar.get_width()/2, v, f"{v:.2f}%",ha="center",va = "bottom", fontsize = 11)
ax.set_ylabel("转化率CAV (%)")
ax.set_title("各环节转化率")
apply_chinese(ax,font_prop)
plt.savefig(FIGURES_DIR/"user_funnel_rates.png")
plt.show()

## 用户行为漏斗分析结论
- 浏览用户 10000，加购用户 8614（86.1%），购买用户 8886（88.9%）
- 购买用户高于收藏用户（6730），说明：
    1. 大量用户跳过收藏、加购后直接购买。
    2. 收藏行为本身使用率低，不适合作为漏斗主链路。
- 建议：漏斗主链改为：【浏览->加购->购买】，收藏单独进行分析
- 进一步分析：需要区分新、老用户，老用户复购会拉高购买转化率。